# Demonstration of Loading and Visualising the Digital Earth Normalised Radar Backscatter Product for Sentinel-1 (Collection 1)

This notebook demonstrates key steps for using Python to load sample data from the Digital Earth Normalised Radar Backscatter Product for Sentinel-1 developed by Geoscience Australia. 
The notebook covers how to load via a STAC API.

## Set Up

### Import required libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from odc.geo import BoundingBox
import odc.geo.xr
import pathlib
import xarray as xr

from pystac_client import Client
from odc.stac import load, configure_s3_access

## Environment set up

In [ ]:
catalog = "https://explorer.dev.dea.ga.gov.au/stac"
stac_client = Client.open(catalog)
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

## Searching and Loading

### Set up area of interest and date range

In [ ]:
aoi_bbox = BoundingBox(
    left=85.08086,
    bottom=-66.47073,
    right=85.21068,
    top=-66.51947,
    crs="EPSG:4326"
)

start_date = "2020-01-01"
end_date = "2026-01-01"

aoi_bbox.explore()

### Set up general data loading settings

In [ ]:
product_to_load = "ga_s1_nrb_iw_hh_1"
measurements_to_load = ["hh_gamma0", "oa_layover_shadow_mask"]
output_crs = "EPSG:3031"

output_res = 20 # metres
group_by_method = "sarard:scene_id"

### Run search through STAC

In [ ]:
stac_items = stac_client.search(
    collections=[product_to_load],
    datetime=f"{start_date}/{end_date}",
    bbox=aoi_bbox.bbox,
).item_collection()

print(f"Found {len(stac_items)} items")

### Lazy-load data

In [ ]:
ds = load(
    items=stac_items,
    bands=measurements_to_load,
    intersects=aoi_bbox.boundary(),
    crs=output_crs,
    resolution=output_res,
    groupby=group_by_method,
    chunks={},
)

In [ ]:
ds

### Load data into memory

In [ ]:
ds = ds.compute()

ds

## Visualise

### Timeseries plot

In [ ]:
ds["hh_gamma0"].plot.imshow(col="time", col_wrap=4, robust=True, cmap="Greys_r")

### Masking

In [ ]:
ds["oa_layover_shadow_mask"].plot.imshow(
    col="time", 
    col_wrap=4)

#### Dilating the mask
May not be needed for coastal regions in Antarctica, shadow and layover generally doesn't appear.


In [ ]:
from de_sar_demo.masking import dilate_mask

# Create a layer with all non-valid pixels, including both layover and shadow
non_valid_pixels = ds["oa_layover_shadow_mask"] != 0

# Create the dilated mask
dilated_mask = dilate_mask(non_valid_pixels, dilation_radius=3)

dilated_mask.plot.imshow(col="time", col_wrap=4)

#### Applying the mask

In [ ]:
ds["hh_gamma0_masked"] = xr.where(dilated_mask==0, ds["hh_gamma0"], np.nan)

ds["hh_gamma0_masked"].plot.imshow(col="time", col_wrap=4, cmap="viridis", robust=True)
plt.show()

## Speckle filtering


In [ ]:
from de_sar_demo.speckle_filters import apply_lee_filter

ds["hh_gamma0_masked_filtered"] = apply_lee_filter(ds["hh_gamma0"], size=5)

ds["hh_gamma0_masked_filtered"].plot.imshow(col="time", col_wrap=4, cmap="Greys_r", robust=True)
plt.show()

In [ ]:
ds["hh_gamma0_masked_filtered_db"] = 10*np.log10(ds["hh_gamma0_masked_filtered"])

ds["hh_gamma0_masked_filtered_db"].plot.imshow(col="time", col_wrap=4, cmap="Greys_r")
plt.show()

## Exporting

In [ ]:
output_path = pathlib.Path("outputs")
output_path.mkdir(exist_ok=True)

In [ ]:
ds_datetime_strings = ds.time.dt.strftime("%Y-%m-%d").values

for timestep in range(len(ds.time)):
    filename = output_path / f"antarctica_hh_filtered_masked_db_{ds_datetime_strings[timestep]}.tif"
    ds["hh_gamma0_masked_filtered_db"].isel(time=timestep).odc.write_cog(filename, overwrite=True)